# Run the lesion segmentation on an OCT scan

Same thing `demo.py` does, one step at a time, so each stage can be seen before
the next runs. No GPU, no training. Two sample scans are included, so it runs as
it stands.

## 1. Setup

The functions come from `demo.py`, so this notebook and the command line run the
same code and cannot disagree.

In [ ]:
import os
import matplotlib.pyplot as plt

# demo.py sits beside this notebook, so it imports directly
import demo

print("models available:", ", ".join(sorted(demo.MODELS)))
print("input geometry  :", demo.INPUT_HEIGHT, "x", demo.INPUT_WIDTH)

## 2. Pick a scan

Two endpoint scans, each with a lesion, and the expert annotation of each.

In [ ]:
SAMPLES = os.path.join('sample_data', 'scans')
TRUTHS = os.path.join('sample_data', 'annotations')

scan_name = 'scan_05.tif'          # change this to use a different scan
scan_path = os.path.join(SAMPLES, scan_name)
truth_path = os.path.join(TRUTHS, scan_name)

scan = demo.load_single_channel(scan_path)
print("loaded", scan_name, "shape", scan.shape)

plt.figure(figsize=(11, 5))
plt.imshow(scan, cmap='gray')
plt.title("The scan as it arrives")
plt.axis('off')
plt.show()

The bright band across the middle is the retina, and the lesion is the disrupted
patch inside it. Most of the frame is not retina at all.

## 3. Predict

Resize to the trained geometry, threshold at 0.5, then keep one connected region,
because one laser burn is one lesion.

In [ ]:
import cv2

model, channels = demo.build_model('plain', None)
resized = cv2.resize(scan, (demo.INPUT_WIDTH, demo.INPUT_HEIGHT))

mask, probability = demo.segment(model, channels, resized, None)

lesion_pixels = int(mask.sum())
frame_pixels = demo.INPUT_HEIGHT * demo.INPUT_WIDTH
print("lesion pixels : %d" % lesion_pixels)
print("share of frame: %.2f%%" % (100.0 * lesion_pixels / frame_pixels))

plt.figure(figsize=(11, 5))
plt.imshow(cv2.cvtColor(demo.draw_overlay(resized, mask), cv2.COLOR_BGR2RGB))
plt.title("Predicted lesion, marked in yellow")
plt.axis('off')
plt.show()

Yellow is what the model calls lesion. That is the whole output when no ground
truth is given.

## 4. Score it

The annotation is a yellow outline drawn on the scan, so it is thresholded and
filled first, the same way the training masks were built.

In [ ]:
truth = demo.load_truth_mask(truth_path, from_annotation=True)
truth = cv2.resize(truth, (demo.INPUT_WIDTH, demo.INPUT_HEIGHT),
                   interpolation=cv2.INTER_NEAREST)

scores = demo.score(mask, truth)
for name in ['dice', 'iou', 'precision', 'recall']:
    print("%-10s %.3f" % (name, scores[name]))
print("%-10s %.1f%% of the lesion" % ('missed', scores['missed']))
print("%-10s %.1f%% of the lesion" % ('extra', scores['extra']))

## 5. See where it went wrong

Yellow is agreement, green is lesion missed, red is tissue wrongly included.
Green against red says which way the model errs, which one score cannot.

In [ ]:
error_map = demo.draw_error_map(resized, mask, truth)

plt.figure(figsize=(13, 6))
plt.imshow(cv2.cvtColor(error_map, cv2.COLOR_BGR2RGB))
plt.title("Yellow correct, green missed lesion, red extra tissue")
plt.axis('off')
plt.show()

## 6. A different model

`plain` takes the image alone; the 2ch models add the residual. Try the harder
scan, `scan_09.tif`, and watch what happens.

In [ ]:
hard_name = 'scan_09.tif'

octf = demo.build_octf()
hybrid, hybrid_channels = demo.build_model('hybrid', octf)

hard_scan = demo.load_single_channel(os.path.join(SAMPLES, hard_name))
hard_resized = cv2.resize(hard_scan, (demo.INPUT_WIDTH, demo.INPUT_HEIGHT))
hard_residual = demo.compute_residual(octf, hard_resized)

hard_truth = demo.load_truth_mask(os.path.join(TRUTHS, hard_name), True)
hard_truth = cv2.resize(hard_truth, (demo.INPUT_WIDTH, demo.INPUT_HEIGHT),
                        interpolation=cv2.INTER_NEAREST)

for label, this_model, this_channels, this_residual in [
        ('plain', model, channels, None),
        ('hybrid', hybrid, hybrid_channels, hard_residual)]:
    this_mask, _ = demo.segment(this_model, this_channels, hard_resized, this_residual)
    this_scores = demo.score(this_mask, hard_truth)
    print("%-8s Dice %.3f   precision %.3f   recall %.3f"
          % (label, this_scores['dice'], this_scores['precision'], this_scores['recall']))

Precision holds while recall collapses: the 2ch model is right about nearly every
pixel it marks and marks far too few. That trade is the study's main finding.

## 7. Your own scan

Point `scan_path` in section 2 at any PNG, JPG, TIFF or BMP. A scanner writes
none of those: export the B-scan you want from its viewer first. Pointed at an
`.e2e`, `.vol`, `.fda` or DICOM, `demo.py` says so and tells you what to do.

These weights were fitted to rat OCT from one study, so on any other source read
the output as a demonstration, not a measurement.